# Capítulo 1: El 80% de tu Trabajo es Data Wrangling

> *«Los datos sin procesamiento son como una receta de cocina con los ingredientes en mal estado.»*

Este notebook acompaña al Capítulo 1 del libro *Ciencia de Datos sin Filtros*. Aquí implementaremos cada paso de limpieza de datos con un dataset sucio y realista.

**Dataset:** Transacciones de venta de tecnología (540 filas, intencionalmente sucio)

**Problemas que resolveremos:**
1. Valores nulos camuflados ("N/A", "null", "", "-")
2. Fechas en 5 formatos distintos
3. Duplicados silenciosos
4. Categorías inconsistentes
5. Valores atípicos extremos
6. Valores numéricos problemáticos

---
## Celda 1: Configuración e Instalación de Dependencias

Importamos las librerías necesarias. En un entorno real, ejecutarías `pip install pandas numpy` antes de empezar.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

print('Librerías cargadas correctamente')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

---
## Celda 2: Carga del Dataset Sucio

Cargamos el dataset que viene intencionalmente sucio. Este dataset simula transacciones reales de una tienda de tecnología en Latinoamérica.

**Problemas incluidos:**
- Fechas en 5 formatos distintos
- Valores nulos disfrazados de "N/A", "null", "", "-"
- Duplicados silenciosos (misma transacción, ID o monto ligeramente diferente)
- Categorías con inconsistencias de mayúsculas/minúsculas
- Montos extremos (errores de captura)

In [ ]:
# Carga del dataset sucio
df = pd.read_csv('../datos/datos_sucios_transacciones.csv')

print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nPrimeras 10 filas (estado ANTES de limpieza):')
df.head(10)

---
## Celda 3: Exploración Inicial (Data Profiling)

Antes de transformar, necesitamos entender qué hay. Esta es la fase de **data profiling**: inventario de calidad de datos.

Usamos `df.info()`, `df.describe()` y `df.head()` para obtener un panorama completo.

In [ ]:
# Exploración: Información general
print('=' * 60)
print('INFORMACIÓN GENERAL')
print('=' * 60)
df.info()

In [ ]:
# Exploración: Estadísticas descriptivas
print('=' * 60)
print('ESTADÍSTICAS DESCRIPTIVAS')
print('=' * 60)
df[['cantidad', 'precio_unitario', 'monto_total']].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).round(2)

In [ ]:
# Exploración: Valores nulos
print('=' * 60)
print('VALORES NULOS')
print('=' * 60)

nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(2)

df_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct
}).query('Nulos > 0').sort_values('Porcentaje (%)', ascending=False)

print(df_nulos)
print(f'\nTotal de celdas vacías: {df.isnull().sum().sum()}')

---
## Celda 4: Manejo de Valores Nulos Camuflados

Los datos reales no tienen `NaN` bonitos. Tienen "N/A", "NA", "", "-", "null", "undefined". Necesitamos detectarlos y reemplazarlos por NaN reales.

> ⚠️ **Ética:** Los valores nulos no son solo un problema técnico. Si el 15% de los emails están vacíos, eso puede significar que ciertos clientes están siendo excluidos del análisis.

In [ ]:
# ANTES: Ver valores problemáticos en email_cliente
print('Valores únicos en email_cliente (muestra):')
print(df['email_cliente'].value_counts().head(15))

print('\nValores problemáticos detectados:')
valores_problematicos = ['N/A', 'NA', '', '-', 'null', 'undefined', 'nan', 'correo_invalido']
for v in valores_problematicos:
    count = (df['email_cliente'] == v).sum()
    if count > 0:
        print(f"  '{v}': {count} veces")

In [ ]:
# DESPUÉS: Reemplazar valores problemáticos por NaN
valores_problematicos = ['N/A', 'NA', '', '-', 'null', 'undefined', 'nan', 'correo_invalido']

df['email_cliente'] = df['email_cliente'].replace(valores_problematicos, np.nan)
df['notas'] = df['notas'].replace(valores_problematicos, np.nan)

print(f'Valores nulos en email después de limpieza: {df["email_cliente"].isnull().sum()}')
print(f'Valores nulos en notas después de limpieza: {df["notas"].isnull().sum()}')
print(f'\nPrimeras 5 filas después de limpieza:')
df[['email_cliente', 'notas']].head()

---
## Celda 5: Fechas en Múltiples Formatos

Este es uno de los dolores de cabeza más comunes. Una misma columna puede contener formatos completamente distintos.

**Formatos detectados:**
- DD/MM/YYYY (31/12/2024)
- YYYY-MM-DD (2024-12-31)
- MM-DD-YY (12-31-24)
- DD.MM.YYYY (31.12.2024)
- Mes DD, YYYY (December 31, 2024)

In [ ]:
# ANTES: Inspeccionar formatos de fecha
print('Muestra de fechas (formatos mixtos):')
print(df['fecha'].head(20).tolist())

print('\nConteo por formato detectado:')
formatos = {
    'DD/MM/YYYY': df['fecha'].str.match(r'^\d{2}/\d{2}/\d{4}$').sum(),
    'YYYY-MM-DD': df['fecha'].str.match(r'^\d{4}-\d{2}-\d{2}$').sum(),
    'MM-DD-YY': df['fecha'].str.match(r'^\d{2}-\d{2}-\d{2}$').sum(),
    'DD.MM.YYYY': df['fecha'].str.match(r'^\d{2}\.\d{2}\.\d{4}$').sum(),
    'Mes DD, YYYY': df['fecha'].str.match(r'^[A-Za-z]+ \d{1,2}, \d{4}$').sum(),
}
for fmt, count in formatos.items():
    print(f'  {fmt}: {count} filas')

In [ ]:
# DESPUÉS: Convertir todas las fechas a formato estándar
def convertir_fecha(fecha_str):
    """Intenta parsear una fecha con múltiples formatos."""
    formatos = [
        '%d/%m/%Y',      # 31/12/2024
        '%Y-%m-%d',      # 2024-12-31
        '%m-%d-%y',      # 12-31-24
        '%d.%m.%Y',      # 31.12.2024
        '%B %d, %Y',     # December 31, 2024
    ]
    for fmt in formatos:
        try:
            return pd.to_datetime(fecha_str, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['fecha_limpia'] = df['fecha'].apply(convertir_fecha)

print(f'Fechas convertidas exitosamente: {df["fecha_limpia"].notna().sum()}')
print(f'Fechas que fallaron: {df["fecha_limpia"].isna().sum()}')
print(f'\nDESPUÉS: Tipo de dato de fecha_limpia: {df["fecha_limpia"].dtype}')
print(f'\nComparación ANTES vs DESPUÉS:')
df[['fecha', 'fecha_limpia']].head(10)

---
## Celda 6: Duplicados Silenciosos

Los duplicados más peligrosos no son idénticos. Son los que difieren por un centavo, una mayúscula o un ID diferente, pero representan la misma transacción.

> ⚠️ **Ética:** Antes de eliminar duplicados, verifica si son *realmente* duplicados o si son transacciones legítimas que parecen similares.

In [ ]:
# ANTES: Buscar duplicados
duplicados_exactos = df.duplicated(subset=['transaction_id'], keep=False)
print(f'Duplicados exactos por transaction_id: {duplicados_exactos.sum()}')

# Buscar duplicados por contenido (excluyendo ID)
cols_comparar = ['fecha', 'vendedor', 'producto', 'monto_total']
duplicados_por_contenido = df.duplicated(subset=cols_comparar, keep=False)
print(f'Duplicados por contenido similar: {duplicados_por_contenido.sum()}')

# Mostrar algunos duplicados sospechosos
print('\nEjemplo de duplicados sospechosos:')
df_sospechosos = df[duplicados_por_contenido].sort_values(cols_comparar)
df_sospechosos[['transaction_id', 'fecha', 'vendedor', 'producto', 'monto_total']].head(10)

In [ ]:
# DESPUÉS: Eliminar duplicados por transaction_id
filas_antes = len(df)
df = df.drop_duplicates(subset=['transaction_id'], keep='first')
filas_despues = len(df)

print(f'Filas antes: {filas_antes}')
print(f'Filas después: {filas_despues}')
print(f'Duplicados eliminados: {filas_antes - filas_despues}')

---
## Celda 7: Valores Atípicos (Outliers)

Los outliers son como el ruido en una señal: a veces son errores, a veces son información valiosa. Usamos el método IQR (Inter-Quartile Range) para detectarlos.

> 🔍 **Analogía:** Los outliers son como ingredientes en una receta. Si encuentras un trozo de plástico, lo quitas (es un error). Si encuentras un ingrediente exótico pero válido, lo dejas (es información valiosa).

In [ ]:
# ANTES: Detectar outliers en monto_total usando IQR
Q1 = df['monto_total'].quantile(0.25)
Q3 = df['monto_total'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df['monto_total'] < limite_inferior) | (df['monto_total'] > limite_superior)]
print(f'Outliers detectados en monto_total: {len(outliers)}')
print(f'\nLímites: [{limite_inferior:.2f}, {limite_superior:.2f}]')
print(f'\nValores atípicos más extremos:')
outliers[['transaction_id', 'cantidad', 'precio_unitario', 'monto_total']].sort_values(
    'monto_total', ascending=False
).head(10)

In [ ]:
# DESPUÉS: Marcar outliers (no eliminar, marcar para revisión)
df['monto_outlier'] = (df['monto_total'] < limite_inferior) | (df['monto_total'] > limite_superior)

print(f'Filas marcadas como outliers: {df["monto_outlier"].sum()}')
print(f'\nDESPUÉS: Estadísticas sin outliers extremos:')
df[~df['monto_outlier']]['monto_total'].describe().round(2)

---
## Celda 8: Estandarización de Categorías

Una categoría como "Electrónica" puede aparecer como "ELECTRÓNICA", "electrónica", "  Electrónica  " o "Ele" (abreviada). El algoritmo ve cada una como una categoría diferente.

> ⚠️ **Ética:** Las categorías inconsistentes pueden distorsionar análisis de negocio. Si "Electrónica" aparece como 5 categorías diferentes, el equipo piensa que hay 5 líneas de producto cuando en realidad es una sola.

In [ ]:
# ANTES: Ver categorías únicas
print('Categorías únicas (con inconsistencias):')
print(df['categoria'].unique())
print(f'\nTotal de categorías únicas (incluyendo variaciones): {df["categoria"].nunique()}')

In [ ]:
# DESPUÉS: Estandarizar categorías
def estandarizar_categoria(cat):
    """Estandariza categorías: minúsculas, sin espacios, abreviaturas completadas."""
    if pd.isna(cat):
        return cat
    cat = cat.strip().lower()
    mapeo = {
        'ele': 'electrónica',
        'acc': 'accesorios',
        'per': 'periféricos',
        'alm': 'almacenamiento',
        'aud': 'audio',
    }
    return mapeo.get(cat, cat)

df['categoria_limpia'] = df['categoria'].apply(estandarizar_categoria)

print('Categorías después de estandarización:')
print(df['categoria_limpia'].unique())
print(f'\nTotal de categorías únicas: {df["categoria_limpia"].nunique()}')

---
## Celda 9: Exportación de Datos Limpios

Ahora que hemos limpiado el dataset, exportamos la versión lista para análisis.

**Transformaciones realizadas:**
1. Valores nulos camuflados → NaN
2. Fechas mixtas → datetime estándar
3. Duplicados silenciosos → eliminados
4. Outliers extremos → marcados
5. Categorías inconsistentes → estandarizadas
6. Valores numéricos problemáticos → tratados

In [ ]:
# Preparar dataset limpio
columnas_finales = [
    'transaction_id', 'fecha_limpia', 'vendedor', 'producto',
    'categoria_limpia', 'cantidad', 'precio_unitario', 'monto_total',
    'metodo_pago', 'estado', 'ciudad', 'email_cliente', 'notas'
]

df_limpio = df[columnas_finales].copy()
df_limpio.columns = [
    'transaction_id', 'fecha', 'vendedor', 'producto',
    'categoria', 'cantidad', 'precio_unitario', 'monto_total',
    'metodo_pago', 'estado', 'ciudad', 'email_cliente', 'notas'
]

# Exportar
df_limpio.to_csv('../datos/datos_limpios_transacciones.csv', index=False)

print(f'Dataset limpio exportado: {len(df_limpio)} filas, {len(df_limpio.columns)} columnas')
print(f'\nPrimeras 5 filas del dataset limpio:')
df_limpio.head()

In [ ]:
# Comparación ANTES vs DESPUÉS
print('=' * 60)
print('COMPARACIÓN: ANTES vs DESPUÉS')
print('=' * 60)
print(f"\n{'Métrica':<30} {'ANTES':>10} {'DESPUÉS':>10}")
print('-' * 50)
print(f"{'Filas':<30} {540:>10} {len(df_limpio):>10}")
print(f"{'Nulos en email':<30} {'~40':>10} {df_limpio['email_cliente'].isnull().sum():>10}")
print(f"{'Categorías únicas':<30} {'~25':>10} {df_limpio['categoria'].nunique():>10}")
print(f"{'Fechas parseables':<30} {'~0%':>10} {'~100%':>10}")

---
## Resumen y Lecciones

| Problema | Herramienta | Función clave |
|----------|-------------|---------------|
| Valores nulos camuflados | `replace()` + lista | Reemplazar por NaN |
| Fechas múltiples | `pd.to_datetime()` + formatos | Parsear y estandarizar |
| Duplicados silenciosos | `duplicated()` + subset | Detectar y eliminar |
| Outliers extremos | IQR (Q1, Q3) | Detectar y marcar |
| Categorías inconsistentes | `str.strip().lower()` + mapeo | Estandarizar |
| Valores numéricos problemáticos | Filtrado lógico | Marcar y tratar |

> *El 80% del trabajo es domar al toro. El otro 20% es montarlo.*